# Phase 4 — NLP Feature Extraction
## Notebook 04.01 — Representation Audit and Row-Level Features

### Goal
Build the first Phase 4 artifact from the current Phase 3 output without assuming that the dataset row count is permanent.

### Scope
This notebook:

- reads `cryptovision_v1_preprocessed.parquet`;
- records the current schema and row count;
- validates `source_row_id` and requested representations;
- compares `text_title_description` and `Filtered_Text` when available;
- creates explainable corpus-independent numeric features;
- saves reports and `row_level_text_features.parquet`.

TF-IDF, BERT, FinBERT, PCA, and TruncatedSVD are not implemented here.

## 1. Imports and repository discovery

### Goal
Load the reusable Stage 1 modules from any notebook launch directory.

### Decision
Repository discovery uses Phase 1 and Phase 3 folders rather than a machine-specific absolute path.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def locate_repository_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "1_data_acquisition").exists()
            and (candidate / "3_text_preprocessing").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Expected Phase 1 and Phase 3 folders."
    )


PROJECT_ROOT = locate_repository_root()
PHASE4_DIR = PROJECT_ROOT / "4_nlp_feature_extraction"
if str(PHASE4_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE4_DIR))

from src.config import load_config
from src.paths import resolve_pipeline_paths
from src.pipeline import run_stage1_pipeline
from src.representation_audit import build_representation_comparison, row_length_metrics
from src.row_features import build_row_level_features
from src.validation import (
    available_representations,
    validate_numeric_feature_output,
    validate_phase3_contract,
)

print("Project root:", PROJECT_ROOT)
print("Phase 4 directory:", PHASE4_DIR)
print("Pandas:", pd.__version__)

Project root: C:\Users\sepehr\PycharmProjects\FinancialNLP
Phase 4 directory: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction
Pandas: 3.0.5


### Result
The notebook resolves paths from the repository itself.

### Interpretation
Portable path discovery makes the notebook reproducible on another machine.

### Decision
Keep filesystem logic in reusable helpers and avoid local absolute paths.

## 2. Load the Stage 1 recipe

### Goal
Inspect the requested representations, validation policy, and output paths before reading data.

### Decision
`reference_row_count` is optional and defaults to `null`. It is never a fixed contract.

In [2]:
CONFIG_PATH = PHASE4_DIR / "configs" / "row_feature_recipe.yaml"
config = load_config(CONFIG_PATH)
paths = resolve_pipeline_paths(PROJECT_ROOT, config)

print("Configuration:", CONFIG_PATH)
print("Phase 3 input:", paths["input"])
print("Optional reference rows:", config["input"].get("reference_row_count"))
print("Requested representations:", config["input"]["representations"])
print("Blocking severities:", config["validation"]["fail_on_severity"])
print("Feature output:", paths["features"])

Configuration: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\configs\row_feature_recipe.yaml
Phase 3 input: C:\Users\sepehr\PycharmProjects\FinancialNLP\3_text_preprocessing\data\processed\cryptovision_v1_preprocessed.parquet
Optional reference rows: None
Requested representations: ['text_title_description', 'Filtered_Text']
Blocking severities: ['critical']
Feature output: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\row_features\row_level_text_features.parquet


### Result
The recipe defines what should be measured, not what the dataset must permanently contain.

### Interpretation
A reference row count can help detect drift, but normal preprocessing changes should not break feature extraction.

### Decision
Record current facts at runtime and reserve failures for structurally impossible runs.

## 3. Read the current Phase 3 Parquet file

### Goal
Load the current processed dataset exactly as Phase 3 produced it.

### Code
No row-count expectation is passed to `read_parquet`.

In [3]:
if not paths["input"].is_file():
    raise FileNotFoundError(
        f"Phase 3 output was not found: {paths['input']}\nRun Phase 3 first."
    )

processed_df = pd.read_parquet(paths["input"])

print("Observed shape:", processed_df.shape)
print("Observed row count:", len(processed_df))
print("Observed column count:", len(processed_df.columns))
display(pd.DataFrame({
    "column": processed_df.columns,
    "dtype": [str(dtype) for dtype in processed_df.dtypes],
}))

Observed shape: (88936, 23)
Observed row count: 88936
Observed column count: 23


,column,dtype
0,source_row_id,int64
1,URL,str
2,Title,str
3,Description,str
4,Full Text,str
5,Date Time,str
6,Coin Type,str
7,Filtered_Text,str
8,sentiment_label,str
9,sentiment_score,float64


### Result
The displayed row count is the current Phase 3 output for this run.

### Interpretation
A changed count may result from improved filtering, duplicate handling, or corrected preprocessing. It is not automatically an error.

### Decision
Store the observed count in reports and metadata instead of hard-coding it in code or documentation.

## 4. Report-first Phase 3 contract validation

### Goal
Check provenance and representation availability while distinguishing warnings from true blockers.

### Validation policy
The following are normally reported without stopping:

- changed row count;
- duplicate or missing identifiers;
- changed identifier dtype or ordering;
- one missing requested representation;
- complete duplicate rows.

The run is blocked only if no useful artifact can be created, such as a missing identifier, an empty dataset, or no available requested representation.

In [4]:
input_cfg = config["input"]
validation_cfg = config["validation"]

contract = validate_phase3_contract(
    processed_df,
    id_column=input_cfg["id_column"],
    requested_representations=input_cfg["representations"],
    reference_row_count=input_cfg.get("reference_row_count"),
    row_count_warning_ratio=validation_cfg["row_count_warning_ratio"],
    fail_on_severity=validation_cfg["fail_on_severity"],
)

display(contract.report)
print("Issues:", contract.issue_count)
print("Warnings:", contract.warning_count)
print("Critical checks:", contract.critical_count)
print("Blocks pipeline:", contract.has_blockers)

# This raises only for checks configured as blocking, which defaults to critical.
contract.raise_for_blockers()

,check,status,severity,blocks_pipeline,observed,expected,message
0,dataset_has_rows,pass,info,False,88936,> 0,The dataset contains rows.
1,row_count_observed,pass,info,False,88936,recorded at runtime,"Row count is descriptive metadata, not a fixed contract."
2,id_column_present,pass,info,False,present,source_row_id,The identifier column is available.
3,representation_present::text_title_description,pass,info,False,present,present,Representation is available.
4,representation_present::Filtered_Text,pass,info,False,present,present,Representation is available.
5,at_least_one_representation_available,pass,info,False,"[text_title_description, Filtered_Text]",at least one requested representation,At least one requested representation can be processed.
6,id_missing_values,pass,info,False,0,0,No identifier values are missing.
7,id_duplicate_rows,pass,info,False,0,0,Identifiers are unique.
8,id_integer_dtype,pass,info,False,int64,integer-like identifier,Identifier dtype is integer.
9,id_monotonic_order,pass,info,False,True,True,Rows are ordered by identifier.


Issues: 0
Warnings: 0
Critical checks: 0
Blocks pipeline: False


### Result
Ordinary changes appear as `info` or `warning` rows. They remain visible but do not prevent the next cells from running.

### Interpretation
Validation should protect data integrity without confusing dataset evolution with corruption.

### Decision
Never repair or delete Phase 3 rows silently inside Phase 4. Report questionable conditions and preserve provenance.

## 5. Select representations available in this run

### Goal
Continue with all requested representations that actually exist.

### Decision
If one representation is missing, Stage 1 can still produce useful features for the other. If none exist, the earlier critical check stops the notebook.

In [5]:
requested_representations = list(input_cfg["representations"])
usable_representations = available_representations(
    processed_df,
    requested_representations,
)
skipped_representations = [
    name for name in requested_representations if name not in usable_representations
]

print("Requested:", requested_representations)
print("Used:", usable_representations)
print("Skipped:", skipped_representations)

Requested: ['text_title_description', 'Filtered_Text']
Used: ['text_title_description', 'Filtered_Text']
Skipped: []


### Result
The list reflects the current dataset schema.

### Interpretation
This fallback keeps the notebook useful during incremental Phase 3 development.

### Decision
Record skipped representations in run metadata rather than pretending they were processed.

## 6. Compare representation coverage and length

### Goal
Measure nulls, empty text, word counts, character counts, the configured percentile, maximum length, and very-short text.

### Code
The same audit function is applied to each available representation.

In [6]:
audit_cfg = config["audit"]
representation_comparison = build_representation_comparison(
    processed_df,
    usable_representations,
    very_short_max_words=audit_cfg["very_short_max_words"],
    very_short_max_characters=audit_cfg["very_short_max_characters"],
    percentile=audit_cfg["percentile"],
)

display(representation_comparison)

,representation,row_count,null_count,null_pct,empty_count,empty_pct,nonempty_count,nonempty_pct,median_word_count,p95_word_count,max_word_count,median_character_count,p95_character_count,max_character_count,very_short_word_count,very_short_word_pct_nonempty,very_short_character_count,very_short_character_pct_nonempty
0,text_title_description,88936,0,0.0000,76,0.0855,88860,99.9145,31.0000,72.0000,958,199.0000,456.0000,5705,0,0.0000,0,0.0000
1,Filtered_Text,88936,1,0.0011,1,0.0011,88935,99.9989,60.0000,318.0000,1636,454.0000,"2,432.0000",12724,530,0.5959,468,0.5262


### Result
Each row summarizes one representation using the same definitions.

### Interpretation
`text_title_description` is project-controlled conservative text. `Filtered_Text` is publisher-provided and may reflect external preprocessing choices.

### Decision
Do not select a universal winner yet. Preserve both candidates for later controlled experiments.

## 7. Inspect representative short and long examples

### Goal
Connect aggregate statistics to actual text rows.

### Decision
Use samples for interpretation only; do not alter the dataset based on individual examples.

In [7]:
for representation in usable_representations:
    metrics = row_length_metrics(processed_df[representation])
    preview = pd.DataFrame({
        input_cfg["id_column"]: processed_df[input_cfg["id_column"]],
        "word_count": metrics["word_count"],
        "character_count": metrics["character_count"],
        representation: processed_df[representation],
    })
    print(f"\nRepresentation: {representation}")
    print("Shortest non-empty examples")
    display(
        preview.loc[preview["character_count"].gt(0)]
        .nsmallest(5, "character_count")
    )
    print("Longest examples")
    display(preview.nlargest(5, "character_count"))


Representation: text_title_description
Shortest non-empty examples


,source_row_id,word_count,character_count,text_title_description
34185,56838,4,23,The Plot Against Crypto
49983,97929,4,24,Why Bitcoin Was Invented
50007,97975,4,25,What Are Fractional NFTs?
6005,9075,5,26,Buy Bitcoin with Bank Card
28383,45054,5,26,The rise of crypto banking


Longest examples


,source_row_id,word_count,character_count,text_title_description
25516,39704,958,5705,"DeFi Comes Roaring Back: Aave, Uniswap, yEarn Lead the Way The DeFi market is making a roaring comeback after two months of losses in the wake of Bitcoin’s ..."
8246,12683,375,2254,"Bulls Will Push BTC Price to $10K Before Leaving the Scene. When Will the Predicted Bitcoin Dump Happen?LTC, ETH, EOS, and XRP Prices Nosedive – Predictions..."
5756,8632,309,1857,No Bitcoin ETF: Cboe Withdraws Proposed Rule Change to List Bitcoin ETFNo Bitcoin ETF: Cboe Withdraws Proposed Rule Change to List Bitcoin ETFNo Bitcoin ETF...
9349,14275,292,1750,"Ethereum Price to Surge to $202, If ETH Breaks above Psychological Level at $185, Crypto Trader BelievesNumber of Bitcoin Holders with Over 1,000 BTC Has Su..."
19481,29731,275,1735,"After 35% Monthly Gains, Stellar (XLM) Looks Ready For The Next Move: Price AnalysisBitcoin Golden Cross Just Happened: What Does It Mean For The BTC Price?..."



Representation: Filtered_Text
Shortest non-empty examples


,source_row_id,word_count,character_count,Filtered_Text
11595,17421,1,2,co
18640,28465,1,3,due
25425,39589,2,3,u.s
25428,39592,2,3,u.s
34419,57386,1,3,btc


Longest examples


,source_row_id,word_count,character_count,Filtered_Text
31303,51075,1636,12724,among retail investor btc often regarded speculative instrument may poised growth future originally designed peer-to-peer electronic cash system word decent...
9117,13932,1537,11993,two year speculation facebook finally unveiled libra cryptocurrency say empower billion user around world giving access financial service providing easy use...
39227,70559,1445,11321,cryptocurrencies utility token security token privacy token digital asset classification multiplying evolving right alongside cryptographic andblockchaintec...
38874,69633,1393,10506,staged powerful rally start punctuated tweets billionaire tesla founder elon musk sent prices soaring joke dogecoin soon though china crackdown cryptocurren...
33763,56022,1413,10348,wake themuch-ballyhooed public listingofcoinbase exchange arguably become theclosest thing household namein cryptocurrency many crypto-newcomers asking expe...


### Result
The previews show what the length statistics mean in real rows.

### Interpretation
Very short text may still contain strong financial signals, while very long text may later require transformer truncation.

### Decision
Keep all rows in Stage 1 and defer model-specific length decisions to the relevant feature recipe.

## 8. Build explainable row-level features

### Goal
Create features that depend only on each row, not on corpus statistics.

### Feature families

- text length and word count;
- numbers and percentages;
- currency symbols;
- uppercase signals;
- question and exclamation marks;
- fixed financial keyword groups.

### Decision
These features are safe to compute before the train/test split because they do not learn parameters from other rows.

In [8]:
feature_cfg = config["features"]
row_features = build_row_level_features(
    processed_df,
    id_column=input_cfg["id_column"],
    representations=usable_representations,
    representation_prefixes=feature_cfg["representation_prefixes"],
    currency_symbols=feature_cfg["currency_symbols"],
    keyword_groups=feature_cfg["keyword_groups"],
)

print("Feature shape:", row_features.shape)
display(row_features.head())

Feature shape: (88936, 49)


,source_row_id,title_description__is_empty,title_description__character_count,title_description__word_count,title_description__number_count,title_description__percentage_count,title_description__currency_symbol_count,title_description__uppercase_character_count,title_description__alphabetic_character_count,title_description__uppercase_ratio,title_description__question_mark_count,title_description__exclamation_mark_count,title_description__keyword_crypto_assets_count,title_description__keyword_crypto_assets_present,title_description__keyword_market_direction_count,title_description__keyword_market_direction_present,title_description__keyword_regulation_count,title_description__keyword_regulation_present,title_description__keyword_instruments_count,title_description__keyword_instruments_present,title_description__keyword_macro_count,title_description__keyword_macro_present,title_description__keyword_risk_events_count,title_description__keyword_risk_events_present,title_description__financial_keyword_count,filtered_text__is_empty,filtered_text__character_count,filtered_text__word_count,filtered_text__number_count,filtered_text__percentage_count,filtered_text__currency_symbol_count,filtered_text__uppercase_character_count,filtered_text__alphabetic_character_count,filtered_text__uppercase_ratio,filtered_text__question_mark_count,filtered_text__exclamation_mark_count,filtered_text__keyword_crypto_assets_count,filtered_text__keyword_crypto_assets_present,filtered_text__keyword_market_direction_count,filtered_text__keyword_market_direction_present,filtered_text__keyword_regulation_count,filtered_text__keyword_regulation_present,filtered_text__keyword_instruments_count,filtered_text__keyword_instruments_present,filtered_text__keyword_macro_count,filtered_text__keyword_macro_present,filtered_text__keyword_risk_events_count,filtered_text__keyword_risk_events_present,filtered_text__financial_keyword_count
0,0,0,194,27,0,0,0,11,164,0.0671,0,0,2,1,0,0,0,0,0,0,0,0,0,0,2,0,93,11,0,0,0,0,83,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,99,12,0,0,0,11,86,0.1279,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,638,74,0,0,0,0,565,0.0000,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1
2,2,0,147,24,1,0,1,9,116,0.0776,0,0,2,1,0,0,0,0,0,0,0,0,0,0,2,0,64,9,0,0,0,0,56,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,0,209,31,0,0,0,15,174,0.0862,1,0,2,1,0,0,0,0,0,0,0,0,0,0,2,0,99,12,0,0,0,0,87,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,0,186,27,2,0,2,10,154,0.0649,0,0,2,1,0,0,0,0,0,0,0,0,0,0,2,0,95,12,0,0,0,0,84,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Result
The output contains `source_row_id` followed by prefixed numeric features for each available representation.

### Interpretation
Prefixing prevents collisions and makes every feature traceable to its source representation.

### Decision
Do not include publisher sentiment, OHLCV, market movement, or future outcomes in this text-only artifact.

## 9. Validate the generated feature artifact

### Goal
Confirm one output row per input row, preserved identifier order, numeric features, and finite values.

### Decision
Identifier duplicates inherited from Phase 3 are warnings. Changed row count or row order in the generated artifact is critical because it would corrupt downstream joins.

In [9]:
feature_validation = validate_numeric_feature_output(
    row_features,
    id_column=input_cfg["id_column"],
    expected_ids=processed_df[input_cfg["id_column"]],
    fail_on_severity=validation_cfg["fail_on_severity"],
)

display(feature_validation.report)
print("Blocks pipeline:", feature_validation.has_blockers)
feature_validation.raise_for_blockers()

,check,status,severity,blocks_pipeline,observed,expected,message
0,feature_id_column_present,pass,info,False,present,source_row_id,The generated artifact must retain the Phase 3 identifier.
1,feature_row_count_matches_input,pass,info,False,88936,88936,Feature extraction must produce exactly one output row per input row.
2,feature_id_order_preserved,pass,info,False,True,True,A changed identifier order would corrupt later joins.
3,feature_id_duplicate_rows,pass,info,False,0,0,Existing duplicate identifiers are reported but do not alter row-local feature computation.
4,feature_columns_exist,pass,info,False,48,> 0,At least one numeric feature column is required.
5,feature_columns_numeric,pass,info,False,all numeric,all feature columns numeric,Only the identifier may be non-feature metadata.
6,feature_nan_values,pass,info,False,0,0,NaN values would make the saved numeric artifact unreliable.
7,feature_values_finite,pass,info,False,True,True,Infinite values are not valid row-level features.


Blocks pipeline: False


### Result
Warnings remain visible. Critical structural failures stop the write step.

### Interpretation
This boundary is stricter than input diagnostics because Phase 4 must not save a feature table with broken row alignment or invalid numeric values.

### Decision
Write only structurally valid features while retaining non-blocking data-quality warnings in reports.

## 10. Run the canonical pipeline and save artifacts

### Goal
Use the tested command-line entry point so notebook and non-notebook execution follow the same policy.

### Outputs

- `row_level_text_features.parquet`;
- `representation_comparison.csv`;
- `representation_comparison.json`;
- `phase3_contract_validation.csv`;
- `row_feature_validation.csv`;
- `stage1_run_metadata.json`.

In [10]:
run_result = run_stage1_pipeline(
    project_root=PROJECT_ROOT,
    config_path=CONFIG_PATH,
)

print("Run status:", run_result["metadata"]["status"])
print("Observed input rows:", run_result["metadata"]["input_rows"])
print("Used representations:", run_result["metadata"]["used_representations"])
print("Skipped representations:", run_result["metadata"]["skipped_representations"])

print("\nGenerated artifacts:")
for name, path in run_result["paths"].items():
    print(f"- {name}: {path}")

display(run_result["contract_validation"])
display(run_result["feature_validation"])
display(run_result["comparison"])

Run status: completed
Observed input rows: 88936
Used representations: ['text_title_description', 'Filtered_Text']
Skipped representations: []

Generated artifacts:
- input: C:\Users\sepehr\PycharmProjects\FinancialNLP\3_text_preprocessing\data\processed\cryptovision_v1_preprocessed.parquet
- features: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\row_features\row_level_text_features.parquet
- comparison_csv: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reports\representation_comparison.csv
- comparison_json: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reports\representation_comparison.json
- contract_validation: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reports\phase3_contract_validation.csv
- feature_validation: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\reports\row_feature_validation.csv
- metadata: C:\Users\sepehr\PycharmProjects

,check,status,severity,blocks_pipeline,observed,expected,message
0,dataset_has_rows,pass,info,False,88936,> 0,The dataset contains rows.
1,row_count_observed,pass,info,False,88936,recorded at runtime,"Row count is descriptive metadata, not a fixed contract."
2,id_column_present,pass,info,False,present,source_row_id,The identifier column is available.
3,representation_present::text_title_description,pass,info,False,present,present,Representation is available.
4,representation_present::Filtered_Text,pass,info,False,present,present,Representation is available.
5,at_least_one_representation_available,pass,info,False,"[text_title_description, Filtered_Text]",at least one requested representation,At least one requested representation can be processed.
6,id_missing_values,pass,info,False,0,0,No identifier values are missing.
7,id_duplicate_rows,pass,info,False,0,0,Identifiers are unique.
8,id_integer_dtype,pass,info,False,int64,integer-like identifier,Identifier dtype is integer.
9,id_monotonic_order,pass,info,False,True,True,Rows are ordered by identifier.


,check,status,severity,blocks_pipeline,observed,expected,message
0,feature_id_column_present,pass,info,False,present,source_row_id,The generated artifact must retain the Phase 3 identifier.
1,feature_row_count_matches_input,pass,info,False,88936,88936,Feature extraction must produce exactly one output row per input row.
2,feature_id_order_preserved,pass,info,False,True,True,A changed identifier order would corrupt later joins.
3,feature_id_duplicate_rows,pass,info,False,0,0,Existing duplicate identifiers are reported but do not alter row-local feature computation.
4,feature_columns_exist,pass,info,False,48,> 0,At least one numeric feature column is required.
5,feature_columns_numeric,pass,info,False,all numeric,all feature columns numeric,Only the identifier may be non-feature metadata.
6,feature_nan_values,pass,info,False,0,0,NaN values would make the saved numeric artifact unreliable.
7,feature_values_finite,pass,info,False,True,True,Infinite values are not valid row-level features.


,representation,row_count,null_count,null_pct,empty_count,empty_pct,nonempty_count,nonempty_pct,median_word_count,p95_word_count,max_word_count,median_character_count,p95_character_count,max_character_count,very_short_word_count,very_short_word_pct_nonempty,very_short_character_count,very_short_character_pct_nonempty
0,text_title_description,88936,0,0.0000,76,0.0855,88860,99.9145,31.0000,72.0000,958,199.0000,456.0000,5705,0,0.0000,0,0.0000
1,Filtered_Text,88936,1,0.0011,1,0.0011,88935,99.9989,60.0000,318.0000,1636,454.0000,"2,432.0000",12724,530,0.5959,468,0.5262


### Result
The metadata records the dataset as it exists now, including its current row count and any warnings.

### Interpretation
A completed run may legitimately have status `completed_with_warnings`. That status means the pipeline produced valid artifacts while preserving issues for review.

### Decision
Treat reports as part of the experiment record rather than converting every difference into an exception.

## 11. Stage 1 conclusions and next steps

### Goal
Define the handoff without implementing corpus-learned methods too early.

### Result
Stage 1 produces row-local numeric features and representation-quality reports tied to `source_row_id`.

### Interpretation
TF-IDF, PCA, and TruncatedSVD learn from collections of rows. Final fitting must occur only after the time-aware split. Frozen BERT and FinBERT embeddings require a separate batching and model-metadata workflow.

### Decision

- Notebook 04.02 will implement Word, Character, and combined TF-IDF recipes with smoke tests only.
- Notebook 04.03 will implement frozen BERT/FinBERT embeddings and PCA/SVD recipes.
- Phase 5 builds event-aligned labels.
- Phase 6 owns time-aware splitting and leakage control.
- Phase 7 joins approved features and labels by `source_row_id`.